# STM32F429 Button Input Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using button inputs on STM32F429 microcontrollers. We'll cover everything from basic button reading to advanced features like debouncing, interrupts, and event detection.

## Table of Contents
1. [Introduction to Button Input](#introduction)
2. [STM32F429 GPIO Overview](#gpio-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic Button Reading](#basic-reading)
6. [Debouncing Techniques](#debouncing)
7. [Interrupt-Driven Buttons](#interrupts)
8. [Button Events and Callbacks](#events)
9. [Advanced Button Features](#advanced)
10. [Multiple Button Management](#multiple-buttons)
11. [Button State Machine](#state-machine)
12. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to Button Input

Button input is one of the most fundamental forms of user interaction in embedded systems. The STM32F429 provides flexible GPIO pins that can be configured for button input with various features.

### Key Button Concepts:
- **Contact Bounce**: Mechanical buttons oscillate when pressed/released
- **Debouncing**: Filtering out contact bounce noise
- **Active Level**: Whether button is active high or active low
- **Pull Resistors**: Internal pull-up or pull-down resistors
- **Interrupts**: Event-driven button detection
- **Polling**: Regular button state checking

### Button Types:
- **Momentary**: Only active while pressed
- **Toggle**: Changes state each press
- **Active Low**: Pressed = 0V, Released = 3.3V
- **Active High**: Pressed = 3.3V, Released = 0V

### Button Events:
- **Pressed**: Button transition to active state
- **Released**: Button transition to inactive state
- **Click**: Complete press-release cycle
- **Double Click**: Two quick clicks
- **Long Press**: Extended button press

### GPIO Register Map (Reference Manual)

The STM32F429 GPIO peripheral uses several registers for configuration:

| Register | Address Offset | Description |
|----------|----------------|-------------|
| GPIOx_MODER | 0x00 | Port mode register |
| GPIOx_OTYPER | 0x04 | Port output type register |
| GPIOx_OSPEEDR | 0x08 | Port output speed register |
| GPIOx_PUPDR | 0x0C | Port pull-up/pull-down register |
| GPIOx_IDR | 0x10 | Port input data register |
| GPIOx_ODR | 0x14 | Port output data register |
| GPIOx_BSRR | 0x18 | Port bit set/reset register |
| GPIOx_LCKR | 0x1C | Port configuration lock register |
| GPIOx_AFRL | 0x20 | Alternate function low register |
| GPIOx_AFRH | 0x24 | Alternate function high register |

### Key GPIO Register Configurations:

#### GPIOx_MODER (Port Mode Register):
- **Bits 2n+1:2n**: MODERy[1:0] - Pin mode
  - 00: Input mode
  - 01: Output mode
  - 10: Alternate function mode
  - 11: Analog mode

#### GPIOx_PUPDR (Pull-up/Pull-down Register):
- **Bits 2n+1:2n**: PUPDRy[1:0] - Pull configuration
  - 00: No pull-up, pull-down
  - 01: Pull-up
  - 10: Pull-down
  - 11: Reserved

#### GPIOx_IDR (Input Data Register):
- **Bit n**: IDRy - Input data bit
  - 0: Input is logic low
  - 1: Input is logic high

### EXTI Register Map (External Interrupts):

| Register | Address Offset | Description |
|----------|----------------|-------------|
| EXTI_IMR | 0x00 | Interrupt mask register |
| EXTI_EMR | 0x04 | Event mask register |
| EXTI_RTSR | 0x08 | Rising trigger selection register |
| EXTI_FTSR | 0x0C | Falling trigger selection register |
| EXTI_SWIER | 0x10 | Software interrupt event register |
| EXTI_PR | 0x14 | Pending register |

#### EXTI_RTSR/FTSR (Trigger Selection):
- **Bit n**: Trigger selection for line n
  - RTSR: Rising edge trigger
  - FTSR: Falling edge trigger

<a id='gpio-overview'></a>
## 2. STM32F429 GPIO Overview

The STM32F429 features multiple GPIO ports (A-E) with 16 pins each, providing flexible input/output capabilities.

### GPIO Features:
- **16 Pins per Port**: GPIOA-GPIOE (80 total pins)
- **Multiple Modes**: Input, output, analog, alternate function
- **Pull Resistors**: Internal pull-up and pull-down
- **Interrupt Capability**: EXTI lines for external interrupts
- **5V Tolerance**: Some pins support 5V input
- **Speed Control**: Configurable output speed

### Pin Naming Convention:
- **PA0-PA15**: Port A pins 0-15
- **PB0-PB15**: Port B pins 0-15
- **PC0-PC15**: Port C pins 0-15
- **PD0-PD15**: Port D pins 0-15
- **PE0-PE15**: Port E pins 0-15

### STM32F429 Discovery Board Buttons:
- **B1 (User Button)**: PA0 (active high, no external pull needed)
- **Joystick**: Multiple pins on different ports
- **Tamper Button**: PC13 (active low)

### GPIO Operating Modes:
- **Input Mode**: Read external signals
- **Output Mode**: Drive external loads
- **Analog Mode**: ADC/DAC operation
- **Alternate Function**: Peripheral pins (UART, SPI, etc.)

### Input Configuration Options:
- **Floating**: No pull resistor
- **Pull-up**: Internal pull-up resistor
- **Pull-down**: Internal pull-down resistor

### Interrupt Lines:
- **EXTI0-EXTI15**: 16 external interrupt lines
- **EXTI16**: PVD output
- **EXTI17**: RTC alarm
- **EXTI18**: USB OTG FS wakeup
- **EXTI19-EXTI22**: Ethernet, USB OTG HS

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- Momentary push buttons
- Connecting wires
- Optional: Pull-up/down resistors

### Basic Button Wiring:

#### Active Low Button (Most Common):
```
STM32 Pin ──┬── Button ─── GND
            │
            └─ 10kΩ Pull-up Resistor ── 3.3V
```
- Pressed: Pin reads 0 (GND)
- Released: Pin reads 1 (3.3V via pull-up)

#### Active High Button:
```
STM32 Pin ──┬── Button ─── 3.3V
            │
            └─ 10kΩ Pull-down Resistor ── GND
```
- Pressed: Pin reads 1 (3.3V)
- Released: Pin reads 0 (GND via pull-down)

### STM32F429 Discovery Board:
- **User Button B1**: PA0 (already connected, active high)
- **Tamper Button**: PC13 (active low with pull-up)

### External Button Connection:
```c
// Example: Connect button to PB1 (active low)
#define BUTTON_PORT    GPIOB
#define BUTTON_PIN     GPIO_PIN_1

// Hardware connections:
// PB1 ──┬── Button ─── GND
//       │
//       └─ 10kΩ Pull-up ── 3.3V
```

### Multiple Buttons:
```c
// Button matrix or individual connections
#define BUTTON1_PORT   GPIOA
#define BUTTON1_PIN    GPIO_PIN_1

#define BUTTON2_PORT   GPIOA  
#define BUTTON2_PIN    GPIO_PIN_2

#define BUTTON3_PORT   GPIOB
#define BUTTON3_PIN    GPIO_PIN_0
```

### Debouncing Hardware:
- **RC Filter**: Resistor + capacitor for hardware debouncing
- **Schmitt Trigger**: Use pins with schmitt trigger input
- **External Debouncer**: Dedicated debouncing IC

### Power Considerations:
- **Current Draw**: Minimal (pull-up resistors)
- **Power Supply**: 3.3V from STM32
- **Ground**: Common ground plane
- **EMI**: Keep button wires short

### ESD Protection:
- **ESD Diodes**: Protect against electrostatic discharge
- **Ground Planes**: Good grounding practices
- **Shielding**: For noisy environments

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Libraries:
```c
#include "button.h"           // Button driver
#include "stm32f4xx_hal.h"    // HAL library
#include "stm32f4xx_hal_gpio.h" // GPIO peripheral
#include "stm32f4xx_hal_exti.h" // External interrupts
```

### Button Configuration Structure:
```c
typedef struct {
    GPIO_TypeDef* port;         // GPIO port (GPIOA, GPIOB, etc.)
    uint16_t pin;               // GPIO pin (GPIO_PIN_0, etc.)
    bool activeLow;             // True for active low buttons
    uint32_t debounceMs;        // Debounce time in milliseconds
    bool enableInterrupt;       // Enable EXTI interrupts
} ButtonConfig_t;
```

### Button Handle Structure:
```c
typedef struct {
    ButtonConfig_t config;      // Button configuration
    ButtonState_t state;        // Current debounced state
    ButtonState_t lastState;    // Previous state
    uint32_t lastChangeTime;    // Last state change timestamp
    bool initialized;           // Initialization flag
} ButtonHandle_t;
```

### Initialization Sequence:
1. **Configure GPIO pin** as input
2. **Set pull resistor** (up/down/none)
3. **Configure EXTI** if using interrupts
4. **Initialize button handle**
5. **Set up debouncing parameters**

### Clock Configuration:
- **GPIO Clock**: Enable port clock (RCC_AHB1ENR)
- **SYSTICK**: For timing (debouncing)
- **EXTI Clock**: Automatically enabled

### Interrupt Configuration:
- **NVIC Priority**: Set interrupt priority
- **EXTI Line**: Configure trigger edges
- **Callback Function**: Handle button events

<a id='basic-reading'></a>
## 5. Basic Button Reading

### Simple Button Reading:
```c
#include "button.h"

// Global button handle
ButtonHandle_t userButton;

void button_example_basic(void) {
    // Initialize button on PA0 (Discovery board user button)
    bool success = Button_Init(&userButton, GPIOA, GPIO_PIN_0);
    if (!success) {
        printf("Button initialization failed!\n");
        return;
    }
    
    printf("Button initialized. Press the user button...\n");
    
    while (1) {
        // Read current button state
        ButtonState_t state = Button_Read(&userButton);
        
        if (state == BUTTON_PRESSED) {
            printf("Button is pressed!\n");
        } else {
            printf("Button is released!\n");
        }
        
        // Small delay to avoid flooding output
        HAL_Delay(100);
    }
}
```

### Raw vs Debounced Reading:
```c
void button_example_raw_vs_debounced(void) {
    ButtonHandle_t button;
    Button_Init(&button, GPIOA, GPIO_PIN_0);
    
    while (1) {
        // Raw reading (no debouncing)
        ButtonState_t raw = Button_ReadRaw(&button);
        
        // Debounced reading
        ButtonState_t debounced = Button_Read(&button);
        
        printf("Raw: %d, Debounced: %d\n", raw, debounced);
        
        HAL_Delay(50);
    }
}
```

### Edge Detection:
```c
void button_example_edge_detection(void) {
    ButtonHandle_t button;
    Button_Init(&button, GPIOA, GPIO_PIN_0);
    
    printf("Press and release the button to see edge detection...\n");
    
    while (1) {
        // Check for press edge
        if (Button_WasPressed(&button)) {
            printf("Button was just pressed!\n");
        }
        
        // Check for release edge
        if (Button_WasReleased(&button)) {
            printf("Button was just released!\n");
        }
        
        HAL_Delay(10); // Small delay for processing
    }
}
```

### Simple Polling Loop:
```c
void button_example_polling(void) {
    ButtonHandle_t button;
    
    // Initialize without interrupts (polling mode)
    ButtonConfig_t config = {
        .port = GPIOA,
        .pin = GPIO_PIN_0,
        .activeLow = false,  // Discovery board button is active high
        .debounceMs = 50,
        .enableInterrupt = false  // Polling mode
    };
    
    bool success = Button_InitCustom(&button, &config);
    if (!success) {
        printf("Button initialization failed!\n");
        return;
    }
    
    printf("Polling for button presses...\n");
    
    while (1) {
        // Check if button is currently pressed
        if (Button_IsPressed(&button)) {
            printf("Button pressed!\n");
            
            // Wait for release
            while (Button_IsPressed(&button)) {
                HAL_Delay(10);
            }
            printf("Button released!\n");
        }
        
        HAL_Delay(10);
    }
}
```

### Understanding Button States:
- **BUTTON_PRESSED**: Button is actively pressed (after debouncing)
- **BUTTON_RELEASED**: Button is released (after debouncing)
- **Raw State**: Immediate GPIO reading (may be noisy)
- **Debounced State**: Filtered state (stable)

### Timing Considerations:
- **Debounce Time**: 20-100ms typically
- **Polling Rate**: 10-100Hz for responsive feel
- **Interrupt Latency**: Minimal delay for interrupt-driven buttons

<a id='debouncing'></a>
## 6. Debouncing Techniques

### Hardware Debouncing:
```c
// RC filter circuit
// Button ── 10kΩ ── GPIO Pin
//           │
//           ├─ 100nF ── GND
//           │
//           └─ Switch ── GND

// This creates a low-pass filter that removes high-frequency bounce
```

### Software Debouncing Methods:

#### Timer-Based Debouncing:
```c
#define DEBOUNCE_TIME_MS 50

typedef struct {
    GPIO_TypeDef* port;
    uint16_t pin;
    bool lastRawState;
    uint32_t lastDebounceTime;
    bool debouncedState;
} DebounceButton_t;

bool ReadDebouncedButton(DebounceButton_t* button) {
    bool rawState = HAL_GPIO_ReadPin(button->port, button->pin);
    
    if (rawState != button->lastRawState) {
        // State changed, reset debounce timer
        button->lastDebounceTime = HAL_GetTick();
        button->lastRawState = rawState;
    } else if ((HAL_GetTick() - button->lastDebounceTime) > DEBOUNCE_TIME_MS) {
        // State stable for debounce time
        if (button->debouncedState != rawState) {
            button->debouncedState = rawState;
        }
    }
    
    return button->debouncedState;
}
```

#### State Machine Debouncing:
```c
typedef enum {
    BUTTON_STATE_IDLE,
    BUTTON_STATE_PRESSED_DEBOUNCE,
    BUTTON_STATE_PRESSED,
    BUTTON_STATE_RELEASED_DEBOUNCE,
    BUTTON_STATE_RELEASED
} ButtonStateMachine_t;

typedef struct {
    GPIO_TypeDef* port;
    uint16_t pin;
    ButtonStateMachine_t state;
    uint32_t timer;
    bool activeLow;
} StateMachineButton_t;

bool ProcessButtonStateMachine(StateMachineButton_t* button) {
    bool rawState = HAL_GPIO_ReadPin(button->port, button->pin);
    if (button->activeLow) rawState = !rawState;  // Invert for active low
    
    uint32_t currentTime = HAL_GetTick();
    
    switch (button->state) {
        case BUTTON_STATE_IDLE:
            if (rawState) {
                button->state = BUTTON_STATE_PRESSED_DEBOUNCE;
                button->timer = currentTime;
            }
            break;
            
        case BUTTON_STATE_PRESSED_DEBOUNCE:
            if (rawState) {
                if ((currentTime - button->timer) >= DEBOUNCE_TIME_MS) {
                    button->state = BUTTON_STATE_PRESSED;
                    return true;  // Button pressed event
                }
            } else {
                button->state = BUTTON_STATE_IDLE;
            }
            break;
            
        case BUTTON_STATE_PRESSED:
            if (!rawState) {
                button->state = BUTTON_STATE_RELEASED_DEBOUNCE;
                button->timer = currentTime;
            }
            break;
            
        case BUTTON_STATE_RELEASED_DEBOUNCE:
            if (!rawState) {
                if ((currentTime - button->timer) >= DEBOUNCE_TIME_MS) {
                    button->state = BUTTON_STATE_IDLE;
                    return false;  // Button released event
                }
            } else {
                button->state = BUTTON_STATE_PRESSED;
            }
            break;
    }
    
    return false;  // No state change
}
```

### Advanced Debouncing:
```c
// Adaptive debouncing based on button characteristics
typedef struct {
    uint32_t pressCount;
    uint32_t releaseCount;
    uint32_t totalPresses;
    uint32_t debounceTime;
} AdaptiveDebounce_t;

void UpdateAdaptiveDebounce(AdaptiveDebounce_t* debounce, bool rawState) {
    if (rawState) {
        debounce->pressCount++;
        debounce->releaseCount = 0;
    } else {
        debounce->releaseCount++;
        debounce->pressCount = 0;
    }
    
    // Adjust debounce time based on button behavior
    if (debounce->pressCount > 10) {
        debounce->debounceTime = 20;  // Fast debounce for clean button
    } else if (debounce->releaseCount > 10) {
        debounce->debounceTime = 100;  // Slow debounce for bouncy button
    }
}
```

### Debouncing Best Practices:
- **Consistent Timing**: Use same debounce time for press and release
- **Test Buttons**: Different buttons have different bounce characteristics
- **Environment**: Temperature and humidity affect bounce
- **Power Supply**: Clean power reduces bounce
- **Mechanical Quality**: Better buttons bounce less

### Common Debouncing Problems:
- **Too Short**: False triggers from bounce
- **Too Long**: Unresponsive feel
- **Asymmetric**: Different debounce for press/release
- **Inconsistent**: Varying debounce time

<a id='interrupts'></a>
## 7. Interrupt-Driven Buttons

### EXTI Interrupt Setup:
```c
// Configure GPIO for interrupt
void ConfigureButtonInterrupt(GPIO_TypeDef* port, uint16_t pin, bool activeLow) {
    GPIO_InitTypeDef gpioInit = {0};
    
    // Enable GPIO clock
    if (port == GPIOA) __HAL_RCC_GPIOA_CLK_ENABLE();
    else if (port == GPIOB) __HAL_RCC_GPIOB_CLK_ENABLE();
    // ... other ports
    
    // Configure pin as interrupt input
    gpioInit.Pin = pin;
    gpioInit.Mode = activeLow ? GPIO_MODE_IT_FALLING : GPIO_MODE_IT_RISING;
    gpioInit.Pull = activeLow ? GPIO_PULLUP : GPIO_PULLDOWN;
    gpioInit.Speed = GPIO_SPEED_FREQ_LOW;
    
    HAL_GPIO_Init(port, &gpioInit);
    
    // Configure EXTI interrupt
    HAL_NVIC_SetPriority(GetEXTIIRQn(pin), 2, 0);
    HAL_NVIC_EnableIRQ(GetEXTIIRQn(pin));
}

// Get EXTI IRQ number for pin
IRQn_Type GetEXTIIRQn(uint16_t pin) {
    if (pin == GPIO_PIN_0) return EXTI0_IRQn;
    if (pin == GPIO_PIN_1) return EXTI1_IRQn;
    if (pin == GPIO_PIN_2) return EXTI2_IRQn;
    if (pin == GPIO_PIN_3) return EXTI3_IRQn;
    if (pin == GPIO_PIN_4) return EXTI4_IRQn;
    if (pin >= GPIO_PIN_5 && pin <= GPIO_PIN_9) return EXTI9_5_IRQn;
    if (pin >= GPIO_PIN_10 && pin <= GPIO_PIN_15) return EXTI15_10_IRQn;
    return EXTI0_IRQn;  // Default
}
```

### Interrupt Handler Implementation:
```c
// Global button handles
ButtonHandle_t button1, button2;

// Interrupt handlers (add to stm32f4xx_it.c)
void EXTI0_IRQHandler(void) {
    HAL_GPIO_EXTI_IRQHandler(GPIO_PIN_0);
}

void EXTI9_5_IRQHandler(void) {
    HAL_GPIO_EXTI_IRQHandler(GPIO_PIN_5);
    HAL_GPIO_EXTI_IRQHandler(GPIO_PIN_6);
    // Handle other pins on this IRQ
}

// EXTI callback function
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == GPIO_PIN_0) {
        // Button 1 interrupt
        ButtonState_t state = Button_Read(&button1);
        if (state == BUTTON_PRESSED) {
            printf("Button 1 pressed (interrupt)\n");
        }
    } else if (GPIO_Pin == GPIO_PIN_5) {
        // Button 2 interrupt
        ButtonState_t state = Button_Read(&button2);
        if (state == BUTTON_PRESSED) {
            printf("Button 2 pressed (interrupt)\n");
        }
    }
}
```

### Interrupt-Driven Button Example:
```c
void button_example_interrupt(void) {
    // Initialize buttons with interrupts enabled
    ButtonConfig_t config1 = {
        .port = GPIOA,
        .pin = GPIO_PIN_0,
        .activeLow = false,
        .debounceMs = 50,
        .enableInterrupt = true
    };
    
    ButtonConfig_t config2 = {
        .port = GPIOB,
        .pin = GPIO_PIN_5,
        .activeLow = true,
        .debounceMs = 50,
        .enableInterrupt = true
    };
    
    bool success1 = Button_InitCustom(&button1, &config1);
    bool success2 = Button_InitCustom(&button2, &config2);
    
    if (!success1 || !success2) {
        printf("Button initialization failed!\n");
        return;
    }
    
    printf("Interrupt-driven buttons initialized.\n");
    printf("Press buttons to see interrupts...\n");
    
    // Main loop can do other tasks
    while (1) {
        // Other processing...
        HAL_Delay(1000);
        printf("Main loop running...\n");
    }
}
```

### Interrupt Configuration Options:
```c
// Both edges (rising and falling)
gpioInit.Mode = GPIO_MODE_IT_RISING_FALLING;

// Rising edge only
gpioInit.Mode = GPIO_MODE_IT_RISING;

// Falling edge only
gpioInit.Mode = GPIO_MODE_IT_FALLING;
```

### Interrupt Priority Management:
```c
// Set different priorities for different buttons
HAL_NVIC_SetPriority(EXTI0_IRQn, 2, 0);    // High priority
HAL_NVIC_SetPriority(EXTI9_5_IRQn, 3, 0);  // Lower priority

// Or use CMSIS functions
NVIC_SetPriority(EXTI0_IRQn, NVIC_EncodePriority(2, 2, 0));
```

### Interrupt Latency Considerations:
- **NVIC Latency**: 12-24 cycles
- **Handler Overhead**: Function call overhead
- **Debouncing**: Still needed in interrupt context
- **Shared IRQs**: Multiple pins per interrupt

### Power Management:
- **Wake from Sleep**: Interrupts can wake MCU
- **Low Power Modes**: Configure for wake-up
- **Interrupt Masking**: Disable during critical sections

<a id='events'></a>
## 8. Button Events and Callbacks

### Event-Driven Button System:
```c
typedef enum {
    BUTTON_EVENT_NONE,
    BUTTON_EVENT_PRESSED,
    BUTTON_EVENT_RELEASED,
    BUTTON_EVENT_CLICK,
    BUTTON_EVENT_DOUBLE_CLICK,
    BUTTON_EVENT_LONG_PRESS,
    BUTTON_EVENT_LONG_RELEASE
} ButtonEvent_t;

// Extended button handle
typedef struct {
    ButtonHandle_t base;
    ButtonEvent_t lastEvent;
    uint32_t pressStartTime;
    uint32_t lastClickTime;
    bool longPressDetected;
    void (*callback)(ButtonEvent_t event);
} EventButtonHandle_t;

// Event processing
void ProcessButtonEvents(EventButtonHandle_t* button) {
    ButtonState_t currentState = Button_Read(&button->base);
    ButtonState_t lastState = button->base.lastState;
    uint32_t currentTime = HAL_GetTick();
    
    // Detect state changes
    if (currentState != lastState) {
        if (currentState == BUTTON_PRESSED) {
            // Button pressed
            button->pressStartTime = currentTime;
            button->lastEvent = BUTTON_EVENT_PRESSED;
            if (button->callback) button->callback(BUTTON_EVENT_PRESSED);
        } else {
            // Button released
            uint32_t pressDuration = currentTime - button->pressStartTime;
            
            if (button->longPressDetected) {
                // Long press release
                button->lastEvent = BUTTON_EVENT_LONG_RELEASE;
                if (button->callback) button->callback(BUTTON_EVENT_LONG_RELEASE);
                button->longPressDetected = false;
            } else {
                // Regular click
                button->lastEvent = BUTTON_EVENT_CLICK;
                if (button->callback) button->callback(BUTTON_EVENT_CLICK);
                
                // Check for double click
                if ((currentTime - button->lastClickTime) < 300) {
                    button->lastEvent = BUTTON_EVENT_DOUBLE_CLICK;
                    if (button->callback) button->callback(BUTTON_EVENT_DOUBLE_CLICK);
                }
                button->lastClickTime = currentTime;
            }
        }
    }
    
    // Check for long press
    if (currentState == BUTTON_PRESSED && 
        !button->longPressDetected && 
        (currentTime - button->pressStartTime) > 1000) {
        
        button->lastEvent = BUTTON_EVENT_LONG_PRESS;
        if (button->callback) button->callback(BUTTON_EVENT_LONG_PRESS);
        button->longPressDetected = true;
    }
}
```

### Callback System Implementation:
```c
// Button callback functions
void Button1Callback(ButtonEvent_t event) {
    switch (event) {
        case BUTTON_EVENT_CLICK:
            printf("Button 1 clicked\n");
            LED_Toggle(LED_GREEN);
            break;
            
        case BUTTON_EVENT_DOUBLE_CLICK:
            printf("Button 1 double-clicked\n");
            LED_Toggle(LED_BLUE);
            break;
            
        case BUTTON_EVENT_LONG_PRESS:
            printf("Button 1 long press started\n");
            LED_On(LED_RED);
            break;
            
        case BUTTON_EVENT_LONG_RELEASE:
            printf("Button 1 long press released\n");
            LED_Off(LED_RED);
            break;
            
        default:
            break;
    }
}

void Button2Callback(ButtonEvent_t event) {
    switch (event) {
        case BUTTON_EVENT_PRESSED:
            printf("Button 2 pressed\n");
            break;
            
        case BUTTON_EVENT_RELEASED:
            printf("Button 2 released\n");
            break;
            
        default:
            break;
    }
}
```

### Event-Driven Main Loop:
```c
EventButtonHandle_t button1, button2;

void button_example_events(void) {
    // Initialize buttons with callbacks
    ButtonConfig_t config1 = {
        .port = GPIOA,
        .pin = GPIO_PIN_0,
        .activeLow = false,
        .debounceMs = 50,
        .enableInterrupt = true
    };
    
    Button_InitCustom(&button1.base, &config1);
    button1.callback = Button1Callback;
    button1.longPressDetected = false;
    button1.lastClickTime = 0;
    
    ButtonConfig_t config2 = {
        .port = GPIOB,
        .pin = GPIO_PIN_5,
        .activeLow = true,
        .debounceMs = 30,
        .enableInterrupt = true
    };
    
    Button_InitCustom(&button2.base, &config2);
    button2.callback = Button2Callback;
    button2.longPressDetected = false;
    button2.lastClickTime = 0;
    
    printf("Event-driven buttons initialized.\n");
    printf("Try different button interactions...\n");
    
    while (1) {
        // Process button events
        ProcessButtonEvents(&button1);
        ProcessButtonEvents(&button2);
        
        // Other application tasks...
        HAL_Delay(10);
    }
}
```

### Advanced Event Features:
- **Gesture Recognition**: Complex button patterns
- **Multi-Tap Detection**: Triple clicks, etc.
- **Hold Time Variation**: Different actions for different hold times
- **Button Combinations**: Multiple button presses
- **Sequence Detection**: Specific button press sequences

### Event Timing Configuration:
```c
#define CLICK_TIMEOUT_MS      300   // Max time between clicks for double-click
#define LONG_PRESS_TIMEOUT_MS 1000  // Time to detect long press
#define MULTI_TAP_TIMEOUT_MS  200   // Time window for multi-tap

typedef struct {
    uint16_t clickTimeout;
    uint16_t longPressTimeout;
    uint16_t multiTapTimeout;
    uint8_t maxTaps;
} ButtonTimingConfig_t;
```

### Event Queue System:
```c
#define EVENT_QUEUE_SIZE 16

typedef struct {
    ButtonEvent_t event;
    uint32_t timestamp;
    uint8_t buttonId;
} ButtonEventQueueItem_t;

typedef struct {
    ButtonEventQueueItem_t queue[EVENT_QUEUE_SIZE];
    uint8_t head;
    uint8_t tail;
    uint8_t count;
} ButtonEventQueue_t;

// Thread-safe event queuing
void QueueButtonEvent(ButtonEventQueue_t* queue, ButtonEvent_t event, uint8_t buttonId) {
    if (queue->count < EVENT_QUEUE_SIZE) {
        queue->queue[queue->head].event = event;
        queue->queue[queue->head].timestamp = HAL_GetTick();
        queue->queue[queue->head].buttonId = buttonId;
        
        queue->head = (queue->head + 1) % EVENT_QUEUE_SIZE;
        queue->count++;
    }
}
```

<a id='advanced'></a>
## 9. Advanced Button Features

### Button Matrix Scanning:
```c
#define MATRIX_ROWS 4
#define MATRIX_COLS 4

typedef struct {
    GPIO_TypeDef* rowPorts[MATRIX_ROWS];
    uint16_t rowPins[MATRIX_ROWS];
    GPIO_TypeDef* colPorts[MATRIX_COLS];
    uint16_t colPins[MATRIX_COLS];
    bool buttonStates[MATRIX_ROWS][MATRIX_COLS];
} ButtonMatrix_t;

void InitButtonMatrix(ButtonMatrix_t* matrix) {
    // Configure row pins as outputs (active high)
    for (int row = 0; row < MATRIX_ROWS; row++) {
        GPIO_InitTypeDef gpioInit = {0};
        gpioInit.Pin = matrix->rowPins[row];
        gpioInit.Mode = GPIO_MODE_OUTPUT_PP;
        gpioInit.Pull = GPIO_NOPULL;
        gpioInit.Speed = GPIO_SPEED_FREQ_LOW;
        HAL_GPIO_Init(matrix->rowPorts[row], &gpioInit);
        
        // Set row low initially
        HAL_GPIO_WritePin(matrix->rowPorts[row], matrix->rowPins[row], GPIO_PIN_RESET);
    }
    
    // Configure column pins as inputs with pull-up
    for (int col = 0; col < MATRIX_COLS; col++) {
        GPIO_InitTypeDef gpioInit = {0};
        gpioInit.Pin = matrix->colPins[col];
        gpioInit.Mode = GPIO_MODE_INPUT;
        gpioInit.Pull = GPIO_PULLUP;
        gpioInit.Speed = GPIO_SPEED_FREQ_LOW;
        HAL_GPIO_Init(matrix->colPorts[col], &gpioInit);
    }
}

void ScanButtonMatrix(ButtonMatrix_t* matrix) {
    for (int row = 0; row < MATRIX_ROWS; row++) {
        // Activate current row (set high)
        HAL_GPIO_WritePin(matrix->rowPorts[row], matrix->rowPins[row], GPIO_PIN_SET);
        
        // Small delay for settling
        HAL_Delay(1);
        
        // Read all columns
        for (int col = 0; col < MATRIX_COLS; col++) {
            GPIO_PinState state = HAL_GPIO_ReadPin(matrix->colPorts[col], matrix->colPins[col]);
            bool pressed = (state == GPIO_PIN_RESET);  // Active low
            
            // Update button state with debouncing
            if (pressed != matrix->buttonStates[row][col]) {
                // State changed, could add debouncing here
                matrix->buttonStates[row][col] = pressed;
                
                if (pressed) {
                    printf("Button pressed: Row %d, Col %d\n", row, col);
                }
            }
        }
        
        // Deactivate row
        HAL_GPIO_WritePin(matrix->rowPorts[row], matrix->rowPins[row], GPIO_PIN_RESET);
    }
}
```

### Capacitive Touch Buttons:
```c
// Simple capacitive sensing using ADC
#define CAPACITIVE_SAMPLES 16

typedef struct {
    ADC_HandleTypeDef* hadc;
    uint32_t channel;
    uint32_t baseline;
    uint32_t threshold;
} CapacitiveButton_t;

void InitCapacitiveButton(CapacitiveButton_t* button, ADC_HandleTypeDef* hadc, uint32_t channel) {
    button->hadc = hadc;
    button->channel = channel;
    button->threshold = 100;  // Adjust based on testing
    
    // Calibrate baseline (no touch)
    button->baseline = 0;
    for (int i = 0; i < CAPACITIVE_SAMPLES; i++) {
        ADC_ChannelConfTypeDef sConfig = {0};
        sConfig.Channel = channel;
        sConfig.Rank = 1;
        sConfig.SamplingTime = ADC_SAMPLETIME_84CYCLES;
        HAL_ADC_ConfigChannel(hadc, &sConfig);
        
        HAL_ADC_Start(hadc);
        HAL_ADC_PollForConversion(hadc, 100);
        button->baseline += HAL_ADC_GetValue(hadc);
        HAL_ADC_Stop(hadc);
        
        HAL_Delay(10);
    }
    button->baseline /= CAPACITIVE_SAMPLES;
}

bool ReadCapacitiveButton(CapacitiveButton_t* button) {
    // Take multiple samples for averaging
    uint32_t sum = 0;
    for (int i = 0; i < CAPACITIVE_SAMPLES; i++) {
        ADC_ChannelConfTypeDef sConfig = {0};
        sConfig.Channel = button->channel;
        sConfig.Rank = 1;
        sConfig.SamplingTime = ADC_SAMPLETIME_84CYCLES;
        HAL_ADC_ConfigChannel(button->hadc, &sConfig);
        
        HAL_ADC_Start(button->hadc);
        HAL_ADC_PollForConversion(button->hadc, 100);
        sum += HAL_ADC_GetValue(button->hadc);
        HAL_ADC_Stop(button->hadc);
    }
    
    uint32_t average = sum / CAPACITIVE_SAMPLES;
    
    // Compare with baseline
    if (average > button->baseline + button->threshold) {
        return true;  // Touch detected
    }
    
    return false;
}
```

### Rotary Encoder with Button:
```c
typedef struct {
    GPIO_TypeDef* aPort;
    uint16_t aPin;
    GPIO_TypeDef* bPort;
    uint16_t bPin;
    ButtonHandle_t button;
    int32_t position;
    int8_t lastState;
} RotaryEncoder_t;

void InitRotaryEncoder(RotaryEncoder_t* encoder) {
    // Configure A and B pins as inputs
    GPIO_InitTypeDef gpioInit = {0};
    gpioInit.Mode = GPIO_MODE_INPUT;
    gpioInit.Pull = GPIO_PULLUP;
    gpioInit.Speed = GPIO_SPEED_FREQ_HIGH;
    
    gpioInit.Pin = encoder->aPin;
    HAL_GPIO_Init(encoder->aPort, &gpioInit);
    
    gpioInit.Pin = encoder->bPin;
    HAL_GPIO_Init(encoder->bPort, &gpioInit);
    
    // Initialize button
    Button_Init(&encoder->button, encoder->aPort, encoder->aPin);  // Using A pin for button
    
    encoder->position = 0;
    encoder->lastState = 0;
}

void UpdateRotaryEncoder(RotaryEncoder_t* encoder) {
    // Read current state
    int8_t a = HAL_GPIO_ReadPin(encoder->aPort, encoder->aPin);
    int8_t b = HAL_GPIO_ReadPin(encoder->bPort, encoder->bPin);
    
    // Combine into 2-bit state
    int8_t currentState = (a << 1) | b;
    
    // Detect state change
    if (currentState != encoder->lastState) {
        // Determine direction using state transition table
        static const int8_t transitions[] = {0, -1, 1, 0, 1, 0, 0, -1, -1, 0, 0, 1, 0, 1, -1, 0};
        int8_t transition = transitions[(encoder->lastState << 2) | currentState];
        
        encoder->position += transition;
        printf("Encoder position: %ld\n", encoder->position);
        
        encoder->lastState = currentState;
    }
    
    // Check button
    if (Button_IsPressed(&encoder->button)) {
        printf("Encoder button pressed\n");
    }
}
```

### Button Statistics and Diagnostics:
```c
typedef struct {
    uint32_t totalPresses;
    uint32_t totalReleases;
    uint32_t bounceEvents;
    uint32_t spuriousPresses;
    uint32_t longPresses;
    uint32_t doubleClicks;
    uint32_t averagePressTime;
    uint32_t minPressTime;
    uint32_t maxPressTime;
} ButtonStats_t;

void UpdateButtonStats(ButtonStats_t* stats, ButtonEvent_t event, uint32_t pressDuration) {
    switch (event) {
        case BUTTON_EVENT_PRESSED:
            stats->totalPresses++;
            break;
            
        case BUTTON_EVENT_RELEASED:
            stats->totalReleases++;
            stats->averagePressTime = (stats->averagePressTime + pressDuration) / 2;
            if (pressDuration < stats->minPressTime) stats->minPressTime = pressDuration;
            if (pressDuration > stats->maxPressTime) stats->maxPressTime = pressDuration;
            break;
            
        case BUTTON_EVENT_LONG_PRESS:
            stats->longPresses++;
            break;
            
        case BUTTON_EVENT_DOUBLE_CLICK:
            stats->doubleClicks++;
            break;
    }
}

void PrintButtonStats(ButtonStats_t* stats) {
    printf("Button Statistics:\n");
    printf("  Total presses: %lu\n", stats->totalPresses);
    printf("  Total releases: %lu\n", stats->totalReleases);
    printf("  Long presses: %lu\n", stats->longPresses);
    printf("  Double clicks: %lu\n", stats->doubleClicks);
    printf("  Average press time: %lu ms\n", stats->averagePressTime);
    printf("  Press time range: %lu-%lu ms\n", stats->minPressTime, stats->maxPressTime);
}
```

### Power Management:
```c
// Configure button for wake-up from sleep
void ConfigureButtonWakeup(GPIO_TypeDef* port, uint16_t pin) {
    // Configure EXTI for wake-up
    HAL_NVIC_EnableIRQ(GetEXTIIRQn(pin));
    
    // Enable wake-up from stop mode
    HAL_PWR_EnableWakeUpPin(PWR_WAKEUP_PIN1);  // Configure based on pin
    
    // Enter sleep mode
    HAL_SuspendTick();
    HAL_PWR_EnterSTOPMode(PWR_LOWPOWERREGULATOR_ON, PWR_STOPENTRY_WFI);
    HAL_ResumeTick();
    
    // System wakes up here on button press
    printf("Woke up from button press!\n");
}
```

<a id='multiple-buttons'></a>
## 10. Multiple Button Management

### Button Manager System:
```c
#define MAX_BUTTONS 8

typedef struct {
    ButtonHandle_t buttons[MAX_BUTTONS];
    uint8_t buttonCount;
    void (*eventCallback)(uint8_t buttonId, ButtonEvent_t event);
} ButtonManager_t;

void ButtonManager_Init(ButtonManager_t* manager) {
    manager->buttonCount = 0;
    manager->eventCallback = NULL;
}

bool ButtonManager_AddButton(ButtonManager_t* manager, 
                           GPIO_TypeDef* port, uint16_t pin, 
                           bool activeLow, uint32_t debounceMs) {
    if (manager->buttonCount >= MAX_BUTTONS) {
        return false;
    }
    
    ButtonConfig_t config = {
        .port = port,
        .pin = pin,
        .activeLow = activeLow,
        .debounceMs = debounceMs,
        .enableInterrupt = true
    };
    
    bool success = Button_InitCustom(&manager->buttons[manager->buttonCount], &config);
    if (success) {
        manager->buttonCount++;
    }
    
    return success;
}

void ButtonManager_Process(ButtonManager_t* manager) {
    for (uint8_t i = 0; i < manager->buttonCount; i++) {
        ButtonState_t currentState = Button_Read(&manager->buttons[i]);
        ButtonState_t lastState = manager->buttons[i].lastState;
        
        // Detect state changes
        if (currentState != lastState) {
            ButtonEvent_t event = (currentState == BUTTON_PRESSED) ? 
                                 BUTTON_EVENT_PRESSED : BUTTON_EVENT_RELEASED;
            
            if (manager->eventCallback) {
                manager->eventCallback(i, event);
            }
        }
    }
}
```

### Multi-Button Example:
```c
ButtonManager_t buttonManager;

void MultiButtonCallback(uint8_t buttonId, ButtonEvent_t event) {
    printf("Button %d: %s\n", buttonId, 
           event == BUTTON_EVENT_PRESSED ? "PRESSED" : "RELEASED");
}

void button_example_multiple(void) {
    // Initialize button manager
    ButtonManager_Init(&buttonManager);
    buttonManager.eventCallback = MultiButtonCallback;
    
    // Add multiple buttons
    ButtonManager_AddButton(&buttonManager, GPIOA, GPIO_PIN_0, false, 50);  // Button 0
    ButtonManager_AddButton(&buttonManager, GPIOB, GPIO_PIN_1, true, 50);   // Button 1
    ButtonManager_AddButton(&buttonManager, GPIOC, GPIO_PIN_2, true, 30);   // Button 2
    
    printf("Multiple button system initialized.\n");
    printf("Press any button to see events...\n");
    
    while (1) {
        // Process all buttons
        ButtonManager_Process(&buttonManager);
        
        // Small delay
        HAL_Delay(10);
    }
}
```

### Button Group Management:
```c
typedef enum {
    BUTTON_GROUP_NAVIGATION,
    BUTTON_GROUP_FUNCTION,
    BUTTON_GROUP_NUMERIC
} ButtonGroup_t;

typedef struct {
    uint8_t buttonIds[8];
    uint8_t buttonCount;
    ButtonGroup_t group;
    void (*groupCallback)(ButtonGroup_t group, uint8_t buttonIndex, ButtonEvent_t event);
} ButtonGroup_t;

void ProcessButtonGroups(ButtonGroup_t* groups, uint8_t groupCount, 
                        ButtonManager_t* manager) {
    // This would integrate with the button manager's event system
    // Each group can have its own processing logic
}

void NavigationGroupCallback(ButtonGroup_t group, uint8_t buttonIndex, ButtonEvent_t event) {
    if (event == BUTTON_EVENT_PRESSED) {
        switch (buttonIndex) {
            case 0: printf("Up\n"); break;
            case 1: printf("Down\n"); break;
            case 2: printf("Left\n"); break;
            case 3: printf("Right\n"); break;
            case 4: printf("Enter\n"); break;
        }
    }
}
```

### Button Layout Mapping:
```c
// Physical button layout to logical functions
typedef enum {
    BTN_UP, BTN_DOWN, BTN_LEFT, BTN_RIGHT,
    BTN_ENTER, BTN_ESCAPE, BTN_MENU,
    BTN_F1, BTN_F2, BTN_F3, BTN_F4
} LogicalButton_t;

typedef struct {
    uint8_t physicalId;
    LogicalButton_t logicalId;
    const char* description;
} ButtonMapping_t;

ButtonMapping_t buttonLayout[] = {
    {0, BTN_UP, "Navigate Up"},
    {1, BTN_DOWN, "Navigate Down"},
    {2, BTN_LEFT, "Navigate Left"},
    {3, BTN_RIGHT, "Navigate Right"},
    {4, BTN_ENTER, "Confirm/Enter"},
    {5, BTN_ESCAPE, "Cancel/Escape"},
    {6, BTN_MENU, "Main Menu"},
    {7, BTN_F1, "Function 1"},
    {8, BTN_F2, "Function 2"}
};

LogicalButton_t MapPhysicalToLogical(uint8_t physicalId) {
    for (int i = 0; i < sizeof(buttonLayout)/sizeof(ButtonMapping_t); i++) {
        if (buttonLayout[i].physicalId == physicalId) {
            return buttonLayout[i].logicalId;
        }
    }
    return BTN_UP;  // Default
}
```

### Simultaneous Button Press Detection:
```c
#define COMBO_TIMEOUT_MS 100

typedef struct {
    uint8_t buttons[3];  // Up to 3-button combos
    uint8_t count;
    uint32_t timestamp;
} ButtonCombo_t;

void DetectButtonCombos(ButtonManager_t* manager, ButtonCombo_t* combo) {
    uint8_t pressedCount = 0;
    uint8_t pressedButtons[8];
    
    // Find all currently pressed buttons
    for (uint8_t i = 0; i < manager->buttonCount; i++) {
        if (Button_IsPressed(&manager->buttons[i])) {
            pressedButtons[pressedCount++] = i;
        }
    }
    
    uint32_t currentTime = HAL_GetTick();
    
    if (pressedCount >= 2) {
        // Multiple buttons pressed
        if ((currentTime - combo->timestamp) < COMBO_TIMEOUT_MS) {
            // Part of existing combo
            for (uint8_t i = 0; i < pressedCount && combo->count < 3; i++) {
                combo->buttons[combo->count++] = pressedButtons[i];
            }
        } else {
            // New combo started
            combo->count = pressedCount;
            for (uint8_t i = 0; i < pressedCount && i < 3; i++) {
                combo->buttons[i] = pressedButtons[i];
            }
            combo->timestamp = currentTime;
        }
    } else if (pressedCount == 0 && combo->count > 0) {
        // Combo completed
        printf("Button combo detected: ");
        for (uint8_t i = 0; i < combo->count; i++) {
            printf("%d ", combo->buttons[i]);
        }
        printf("\n");
        
        // Process combo
        ProcessButtonCombo(combo);
        
        // Reset combo
        combo->count = 0;
    }
}
```

<a id='state-machine'></a>
## 11. Button State Machine

### Comprehensive State Machine:
```c
typedef enum {
    BTN_SM_IDLE,
    BTN_SM_DEBOUNCE_PRESS,
    BTN_SM_PRESSED,
    BTN_SM_DEBOUNCE_RELEASE,
    BTN_SM_CLICK,
    BTN_SM_DOUBLE_CLICK_WAIT,
    BTN_SM_DOUBLE_CLICK,
    BTN_SM_LONG_PRESS_START,
    BTN_SM_LONG_PRESS_HOLD,
    BTN_SM_LONG_PRESS_RELEASE
} ButtonStateMachineState_t;

typedef struct {
    ButtonHandle_t button;
    ButtonStateMachineState_t state;
    uint32_t stateTimer;
    uint32_t lastClickTime;
    uint8_t clickCount;
    void (*eventCallback)(ButtonEvent_t event);
} ButtonStateMachine_t;

void ButtonStateMachine_Init(ButtonStateMachine_t* sm, 
                           GPIO_TypeDef* port, uint16_t pin,
                           void (*callback)(ButtonEvent_t)) {
    ButtonConfig_t config = {
        .port = port,
        .pin = pin,
        .activeLow = false,
        .debounceMs = 50,
        .enableInterrupt = true
    };
    
    Button_InitCustom(&sm->button, &config);
    sm->state = BTN_SM_IDLE;
    sm->stateTimer = 0;
    sm->lastClickTime = 0;
    sm->clickCount = 0;
    sm->eventCallback = callback;
}

void ButtonStateMachine_Process(ButtonStateMachine_t* sm) {
    ButtonState_t rawState = Button_ReadRaw(&sm->button);
    uint32_t currentTime = HAL_GetTick();
    
    // State machine logic
    switch (sm->state) {
        case BTN_SM_IDLE:
            if (rawState == BUTTON_PRESSED) {
                sm->state = BTN_SM_DEBOUNCE_PRESS;
                sm->stateTimer = currentTime;
            }
            break;
            
        case BTN_SM_DEBOUNCE_PRESS:
            if (rawState == BUTTON_PRESSED) {
                if ((currentTime - sm->stateTimer) >= sm->button.config.debounceMs) {
                    sm->state = BTN_SM_PRESSED;
                    sm->stateTimer = currentTime;
                    if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_PRESSED);
                }
            } else {
                sm->state = BTN_SM_IDLE;
            }
            break;
            
        case BTN_SM_PRESSED:
            if (rawState == BUTTON_RELEASED) {
                sm->state = BTN_SM_DEBOUNCE_RELEASE;
                sm->stateTimer = currentTime;
            } else if ((currentTime - sm->stateTimer) > 1000) {
                sm->state = BTN_SM_LONG_PRESS_START;
                if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_LONG_PRESS);
            }
            break;
            
        case BTN_SM_DEBOUNCE_RELEASE:
            if (rawState == BUTTON_RELEASED) {
                if ((currentTime - sm->stateTimer) >= sm->button.config.debounceMs) {
                    sm->state = BTN_SM_CLICK;
                    sm->clickCount++;
                    sm->stateTimer = currentTime;
                }
            } else {
                sm->state = BTN_SM_PRESSED;
            }
            break;
            
        case BTN_SM_CLICK:
            if ((currentTime - sm->stateTimer) < 300) {
                // Wait for possible double click
                if (rawState == BUTTON_PRESSED) {
                    sm->state = BTN_SM_DEBOUNCE_PRESS;
                    sm->stateTimer = currentTime;
                }
            } else {
                // Single click timeout
                if (sm->clickCount == 1) {
                    if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_CLICK);
                } else if (sm->clickCount == 2) {
                    if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_DOUBLE_CLICK);
                }
                sm->clickCount = 0;
                sm->state = BTN_SM_IDLE;
            }
            break;
            
        case BTN_SM_LONG_PRESS_START:
            if (rawState == BUTTON_RELEASED) {
                sm->state = BTN_SM_LONG_PRESS_RELEASE;
                if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_LONG_RELEASE);
            } else {
                sm->state = BTN_SM_LONG_PRESS_HOLD;
            }
            break;
            
        case BTN_SM_LONG_PRESS_HOLD:
            if (rawState == BUTTON_RELEASED) {
                sm->state = BTN_SM_LONG_PRESS_RELEASE;
                if (sm->eventCallback) sm->eventCallback(BUTTON_EVENT_LONG_RELEASE);
            }
            // Stay in hold state
            break;
            
        case BTN_SM_LONG_PRESS_RELEASE:
            sm->state = BTN_SM_IDLE;
            break;
            
        default:
            sm->state = BTN_SM_IDLE;
            break;
    }
}
```

### State Machine Example:
```c
void ButtonEventCallback(ButtonEvent_t event) {
    switch (event) {
        case BUTTON_EVENT_PRESSED:
            printf("Button pressed\n");
            LED_On(LED_BLUE);
            break;
            
        case BUTTON_EVENT_CLICK:
            printf("Single click\n");
            LED_Toggle(LED_GREEN);
            break;
            
        case BUTTON_EVENT_DOUBLE_CLICK:
            printf("Double click\n");
            LED_Toggle(LED_RED);
            break;
            
        case BUTTON_EVENT_LONG_PRESS:
            printf("Long press started\n");
            LED_On(LED_ORANGE);
            break;
            
        case BUTTON_EVENT_LONG_RELEASE:
            printf("Long press released\n");
            LED_Off(LED_ORANGE);
            break;
            
        default:
            break;
    }
}

void button_example_state_machine(void) {
    ButtonStateMachine_t buttonSM;
    
    ButtonStateMachine_Init(&buttonSM, GPIOA, GPIO_PIN_0, ButtonEventCallback);
    
    printf("State machine button initialized.\n");
    printf("Try: single click, double click, long press...\n");
    
    while (1) {
        ButtonStateMachine_Process(&buttonSM);
        HAL_Delay(10);
    }
}
```

### State Machine Advantages:
- **Predictable Behavior**: Clear state transitions
- **Robust**: Handles all edge cases
- **Extensible**: Easy to add new states/events
- **Debuggable**: Clear state visibility
- **Maintainable**: Well-structured code

### State Machine Debugging:
```c
const char* GetStateName(ButtonStateMachineState_t state) {
    switch (state) {
        case BTN_SM_IDLE: return "IDLE";
        case BTN_SM_DEBOUNCE_PRESS: return "DEBOUNCE_PRESS";
        case BTN_SM_PRESSED: return "PRESSED";
        case BTN_SM_DEBOUNCE_RELEASE: return "DEBOUNCE_RELEASE";
        case BTN_SM_CLICK: return "CLICK";
        case BTN_SM_DOUBLE_CLICK_WAIT: return "DOUBLE_CLICK_WAIT";
        case BTN_SM_DOUBLE_CLICK: return "DOUBLE_CLICK";
        case BTN_SM_LONG_PRESS_START: return "LONG_PRESS_START";
        case BTN_SM_LONG_PRESS_HOLD: return "LONG_PRESS_HOLD";
        case BTN_SM_LONG_PRESS_RELEASE: return "LONG_PRESS_RELEASE";
        default: return "UNKNOWN";
    }
}

void DebugButtonStateMachine(ButtonStateMachine_t* sm) {
    static ButtonStateMachineState_t lastState = BTN_SM_IDLE;
    
    if (sm->state != lastState) {
        printf("State: %s -> %s\n", 
               GetStateName(lastState), 
               GetStateName(sm->state));
        lastState = sm->state;
    }
}
```

### Advanced State Machines:
- **Hierarchical States**: Parent/child state relationships
- **Parallel States**: Multiple simultaneous state machines
- **Event Queues**: Asynchronous event processing
- **State History**: Undo/redo functionality
- **State Persistence**: Save/restore state across power cycles